<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_102_embedding_geometry_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 📐 **Module 102 — Embedding Geometry Lab** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 15 · Mechanistic Interpretability Workshop (Cohen ML4LLM deep-dives)


# Module 102 — Embedding Geometry Lab

> Embeddings are the substrate of every modern NLP system (M30 RAG, M53 vector DB, M65 multimodal). Cohen Ch3 (projects 8-15) treats them as a **geometric object** — distances, angles, axes — and shows you how the same word, in different models, lives in *different geometries*. This module rebuilds that intuition with the 2026 toolkit: **all-to-all cosine similarity heatmaps · sequential next-token similarity · cosine over networks of related concepts · RSA (Representational Similarity Analysis) across GPT-2 vs BERT vs Llama-3 vs Qwen · linear semantic axes (gender, sentiment, size) · the king−man+woman analogy revisited · isotropy + the "anisotropy problem" + alignment with CLIP / SigLIP**.

### What you'll cover
1. What an **embedding** actually is — Token Embedding vs **Contextual Embedding**
2. **All-to-all cosine similarity** — the matrix that explains everything
3. **Sequential cosine** — how similarity changes word-by-word
4. **Cosine graphs** — turning a similarity matrix into a network
5. **RSA across models** — comparing GPT-2 to BERT to Llama-3
6. **Linear semantic axes** — gender, polarity, formality directions
7. **Analogy vectors** — the king−man+woman test (and why it sometimes fails)
8. **Anisotropy** + **whitening** — why all embeddings look similar by default
9. **Aligning embeddings** across models — Procrustes, CCA, orthogonal mappers
10. The **2026 stack** — production embedding choices and how to debug them


## 1 · Two flavors of embedding — token vs contextual

| Type | What it is | Example |
|---|---|---|
| **Token embedding** (`E_t`) | A learned `[V × D]` table; `embedding[token_id]` gives a vector that **does not depend on context** | GPT-2 wte, Word2Vec, GloVe |
| **Contextual embedding** (`H_t^{(l)}`) | The activation after **layer `l`** for token `t`; **depends on every other token** | The output of any Transformer layer |
| **Output projection** (often shared with `E_t`) | A `[V × D]` matrix used to project the final hidden state to logits | M19/M20 callback — tied embeddings |
| **Pooled sentence / passage embedding** | A single vector per text; usually `mean(H_last)` or `[CLS]` | OpenAI text-embedding-3, BGE, E5, Nomic, Mistral-Embed |

**Token embeddings** are static across the entire model lifetime. **Contextual embeddings** change with context, layer, and model state. Cohen Ch3 spends most time on token embeddings (Word2Vec-style analysis) because they're easier to reason about; this module covers both.

> 🧱 **Why this matters.** When people say "BERT vs GPT-2 embeddings," they usually mean *one specific layer's contextual activations*. RAG embedding models (BGE, E5, Mistral-Embed) are *contextual-mean-pooled with a contrastive head*. Knowing which layer / pooling you're using is half of debugging an embedding bug.


## 2 · All-to-all cosine similarity

Cohen proj 8 builds the simplest possible mech-interp object: take a list of `N` words, embed each, compute the `N×N` matrix `S_{ij} = cos(e_i, e_j)`.

```
            cat    dog    car    truck  joy   sadness
cat       [ 1.00  0.71  0.21   0.20   0.18   0.15  ]
dog       [ 0.71  1.00  0.23   0.22   0.20   0.16  ]
car       [ 0.21  0.23  1.00   0.81   0.07   0.06  ]
truck     [ 0.20  0.22  0.81   1.00   0.06   0.05  ]
joy       [ 0.18  0.20  0.07   0.06   1.00   0.62  ]
sadness   [ 0.15  0.16  0.06   0.05   0.62   1.00  ]
```

**Block-diagonal structure** = semantic categories. cat-dog cluster; car-truck cluster; joy-sadness cluster. The clusters are *not* labeled — they emerged from the embedding training (Cohen calls this **unsupervised category discovery**).

Two findings:
1. **Cosine sim of any two random English words ≈ 0.15-0.30**, not 0. That's the **anisotropy** problem (Section 8).
2. **Within-cluster sims plateau** around 0.6-0.8 — you rarely see 0.95+ except for synonyms and morphological variants ("run", "running", "ran").


In [ ]:
# Cohen proj 8 — all-to-all cosine similarity on token embeddings
import torch, torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("gpt2")
tok   = AutoTokenizer.from_pretrained("gpt2")

words = ["cat", "dog", "lion", "car", "truck", "bicycle",
         "joy", "sadness", "anger", "Paris", "Tokyo", "Mumbai"]

# token embedding (NOT contextual) — wte = word-token-embedding
E = model.wte.weight                                # [V, D]
ids = [tok.encode(" " + w, add_special_tokens=False)[0] for w in words]  # leading space matters!
V = E[ids]                                          # [N, D]

S = F.cosine_similarity(V[:, None, :], V[None, :, :], dim=-1)   # [N, N]
print(S.round(decimals=2))


## 3 · Sequential cosine similarity

Cohen proj 9-10 walks through a sentence one word at a time, computing the cosine similarity between the **current** and **previous** contextual embeddings at a chosen layer.

```
"the   cat   sat   on   the   mat"
        ↓     ↓    ↓     ↓     ↓
sims:  0.42 0.51 0.55 0.44 0.49      (between adjacent words)
```

Patterns to look for:
- **Spikes at content words** (cat, sat, mat) — semantic concentration
- **Dips at function words** (the, on) — they "blend" with their neighbors
- **Step-up at named entities** — "New York" should have a sharp internal similarity peak
- **Drop at sentence boundaries** — periods and `<EOS>` are usually outliers

Cohen proj 10 extends this to *numbers* — `"1 2 3 4 5"` — and you see a much more *uniform* similarity, because numeric tokens form a regular ladder in embedding space. **The ladder structure is a learned numeracy signal** — useful for arithmetic prompting.

> 🧠 **A mechanistic insight.** Function words (the, of, is, a) have lower **isotropy** because they show up in *every* context — their average embedding ends up near the origin and their per-context activations are dominated by attention rather than self-content. Content words have higher isotropy. This is one of the underlying reasons frequency-based reweighting helps retrieval (BM25, BGE-reranker).


## 4 · Cosine graphs — networks of related concepts

Cohen proj 11: take the cosine-similarity matrix, **threshold** it at some value (say 0.5), and treat the result as the **adjacency matrix of a graph**. Now you have a graph of concepts; edges connect semantically close pairs.

You can:
- **Cluster** with Louvain / Leiden (M87 GraphRAG callback)
- **Visualize** with NetworkX / Gephi / PyVis
- **Run GNNs** on this graph for downstream tasks (M96 callback)

This is exactly how **knowledge graphs** for retrieval and **semantic ontologies** are bootstrapped today. The 2026 production pipeline:

```
documents → contextual embeddings → cosine graph at threshold τ
         → Leiden communities → community summaries (LLM)
         → GraphRAG (M87)
```

Tuning `τ` is the key knob: too high → disconnected concepts; too low → one mega-cluster. Most production systems use **percentile-based thresholding** (keep top-5% of edges per node) rather than a global threshold.


## 5 · RSA — comparing two models' geometries

**Representational Similarity Analysis** (Kriegeskorte 2008, neuroscience → Cohen 2024 applied to LLMs). The trick: instead of comparing raw `[N × D]` embeddings (which is impossible across models with different `D`), compare the **similarity matrices** they induce on the *same* `N` stimuli.

```
Step 1 (per model): compute S_M = N × N cosine matrix on stimuli
Step 2: correlate the upper triangle of S_GPT2 vs S_BERT (Pearson or Spearman)
        → one number ∈ [-1, 1] telling you how "similar" the two geometries are
```

Cohen proj 12 runs this on `N=200` words across GPT-2 and BERT. Result: Pearson `r ≈ 0.45-0.55` — the two models agree on broad categories (animals vs vehicles vs emotions) but disagree on fine-grained relations.

**Modern RSA findings (2024-2026):**
- GPT-2 vs Llama-3: `r ≈ 0.65-0.75` (same arch family)
- BERT vs Llama-3: `r ≈ 0.40-0.50`
- GPT-2 vs CLIP-text: `r ≈ 0.50` (CLIP is pulled toward visual concepts)
- GPT-2 vs Whisper-encoder: `r ≈ 0.20` (different modality)

RSA gives an architecture-agnostic way to ask: *"how similar are these two latent spaces?"* It works across language, vision, audio — the only requirement is a common set of stimuli.

> 📊 **Why RSA matters for production.** When you swap from `text-embedding-3-small` to `text-embedding-3-large`, all your vector-DB neighbors change. RSA on a held-out corpus tells you *by how much*, so you can decide whether to re-rank, re-embed, or live with the drift.


## 6 · Linear semantic axes

Cohen proj 14: take pairs of opposites (`he/she`, `king/queen`, `happy/sad`, `big/small`) and *subtract* their embeddings to get **direction vectors**. Then project new words onto those axes.

```
gender_axis = E["he"] - E["she"]                           # or mean over pairs
            ≈ "+masculine" direction in embedding space

score_M = cosine(E["doctor"], gender_axis)                  # +0.18 in GPT-2 token emb
score_M = cosine(E["nurse"],  gender_axis)                  # -0.21
```

You can build a **basis of semantic axes**: gender, sentiment (joy/sad), size (big/small), age (young/old), formality, intensity. Project any new word into this 5-D semantic frame.

**Bias side (M100 callback).** This is *exactly* how Bolukbasi 2016 ("Man is to Computer Programmer as Woman is to Homemaker?") quantified gender bias in word embeddings. Modern measurement uses **SEAT** (Sentence-Encoder Association Test, May 2019) and **WEAT** for token-level. **Llama-3 Instruct shows ~30-50% less gender axis projection** for occupations than GPT-2; aligning with CAI (M89) further reduces it.

> 🪟 **A diagnostic for fine-tuning.** Run the gender / sentiment / formality axis on your fine-tuned model vs the base. Drift indicates the SFT data is pulling the model in a particular direction — sometimes desirable (chatty), sometimes not (biased).


## 7 · Analogy vectors — the king−man+woman test

Mikolov 2013 showed `E[king] − E[man] + E[woman] ≈ E[queen]`. Cohen proj 15 walks the modern version: it works **sometimes**, fails **often**, and the failures are interpretable.

| Analogy | Top-3 nearest |
|---|---|
| king − man + woman | **queen** ✅ · monarch · princess |
| Paris − France + Italy | **Rome** ✅ · Milan · Naples |
| big − small + tall | **short** ✅ · taller · towering |
| good − bad + cold | **hot** ✅ · warm · cool |
| dog − bark + cat | meow ✅ · purr · cat (cheating!) |
| pizza − Italy + Japan | sushi ✅ · ramen · pizza |
| programmer − he + she | nurse · teacher · programmer (mixed — bias surfaces here) |

**Why it sometimes fails:**
1. **The closest vector is often the input itself** (`E[man]`); modern analogy code subtracts `{man, king}` from candidates
2. **Function-word contamination** — leading-space variants vs no-space cause noise
3. **Anisotropy** (Section 8) crushes everything toward an average direction
4. **Polysemy** — "bank" lives in multiple places

The 2024 paper *"Analogies, but bigger"* showed that contextual embeddings (Llama-3 layer 12) **outperform** token embeddings on analogies — but you have to choose the *right* layer.


## 8 · Anisotropy — why all embeddings look similar by default

Empirically, **random pairs of LLM contextual embeddings have cosine ~0.3-0.7** instead of ~0 (Ethayarajh 2019). This **anisotropy** comes from a small number of dominant directions in the residual stream — the "**outlier features**" Sun & Dettmers found in 2024.

| Model | Random-pair cos | Reason |
|---|---|---|
| GPT-2 layer 12 | ~0.35 | Position embeddings dominate |
| BERT layer 12 | ~0.45 | High-magnitude "constants" in some dimensions |
| Llama-3 layer 16 | ~0.20 | RMSNorm helps |
| BGE-large (post-contrastive) | ~0.05 | Contrastive training enforces uniformity |
| OpenAI text-embedding-3 | ~0.05 | Same story |

**Fixes** (post-hoc, no retraining):
- **Whitening / ZCA** — center, decorrelate, scale to unit variance
- **All-but-the-top** (Mu 2018) — subtract first `k` principal components
- **BERT-flow / SimCSE / Contrastive fine-tune** — train the model to spread embeddings out

If you're building a retrieval system and your top-1 recall is bad, **the first thing to test is anisotropy**. Whitening alone can lift recall by 5-15 points on STS benchmarks.

> 🧊 **The 2024-2026 production reality.** Modern embedding models (`text-embedding-3`, `bge-m3`, `nomic-embed-v1.5`, `voyage-3`) are all contrastively post-trained — anisotropy is largely solved out-of-the-box. But if you're using *raw* model activations (e.g. for mech-interp), expect anisotropy and whiten before any analysis.


In [ ]:
# Anisotropy diagnostic + whitening fix
import torch
from transformers import AutoModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
m   = AutoModel.from_pretrained("gpt2", output_hidden_states=True).eval()

texts = ["a random sentence about cats.", "another sentence about trucks.",
         "joy is a feeling.", "Paris is beautiful in summer."]
E = []
for t in texts:
    ids = tok(t, return_tensors="pt").input_ids
    with torch.no_grad():
        h = m(ids).last_hidden_state.mean(1)        # mean-pool last layer
    E.append(h[0])
E = torch.stack(E)                                    # [N, D]

# random-pair cosine — anisotropy diagnostic
import torch.nn.functional as F
print("raw cosines:", F.cosine_similarity(E[None], E[:, None], dim=-1))

# whitening
E_white = E - E.mean(0, keepdim=True)
U, S, V = torch.svd(E_white)
E_white = E_white @ V / (S + 1e-6)
print("whitened cosines:", F.cosine_similarity(E_white[None], E_white[:, None], dim=-1))


## 9 · Aligning embeddings across models

Swap from `text-embedding-3-small` to `voyage-3`? **Indices don't translate.** Three techniques to align two embedding spaces given a shared anchor set of N items:

| Method | Idea | When |
|---|---|---|
| **Procrustes alignment** | `argmin_Q ||A_1 − A_2·Q||_F` subject to `Q^TQ=I` (orthogonal rotation) | Same-dim spaces; cheap + closed-form |
| **CCA / Deep CCA** | Maximize correlation between projections | Different dims; needs more anchor points |
| **Linear regression** | `argmin_W ||A_1 − A_2·W||_F` (unconstrained) | Most flexible; overfits with small anchors |
| **MLP head** | Learn a non-linear mapping | When linear alignment isn't good enough |
| **Transformer encoder reranker** | Cross-encoder on top of both | Best quality, slowest |

**Industrial use cases:**
- **Embedding migration** without re-embedding 100M documents
- **Multi-lingual retrieval** with monolingual embedders (LASER, LaBSE)
- **Cross-modal alignment** — CLIP's image-text contrastive is a special case
- **Federated learning** — different clients embed differently; align at the server

Procrustes alignment is 5 lines of NumPy and is the right starting point for almost any embedding-migration project.


## 10 · The 2026 stack — production embeddings + debugging

| Tier | Closed | Open |
|---|---|---|
| **Top quality** | OpenAI `text-embedding-3-large` (3072-D) · Voyage-3 (1024) · Cohere `embed-multilingual-v3` | **BGE-M3** (1024, multilingual + sparse + multi-vector) · **NV-Embed-v2** (4096) · **Stella-1.5B** |
| **Best per-token cost** | OpenAI `text-embedding-3-small` (1536, $0.02 / 1M tok) | **Nomic-Embed-v1.5** (768) · **BGE-base-en-v1.5** (768) |
| **Edge / on-device (M90)** | — | **MiniLM-L6-v2** (384) · **GTE-small** (384) · **Snowflake-Arctic-embed-xs** |
| **Multimodal** | OpenAI multimodal embed (Q1 2026) · Google multimodal | **CLIP-ViT-L/14** · **SigLIP-So400m** · **JINA-Embeddings-V3** · **VLM2Vec** |
| **Long-context** | Voyage-3-Large (32k) · Cohere-v4 | **BGE-Long** (8k) · **Nomic-Embed-Vision** (8k) · **Jina-Embeddings-V3** (8k) |

**Production debugging checklist** when retrieval breaks:
1. **Anisotropy diagnostic** — is mean cosine of random pairs > 0.3? Whiten.
2. **Embedding dimensionality** — did you truncate (Matryoshka) too aggressively?
3. **Normalization** — vector store expects L2-normalized? Some embedders ship pre-normalized, some don't.
4. **Tokenizer mismatch** — same tokenizer used at indexing vs query?
5. **Truncation** — long documents may have been silently cut.
6. **Pooling** — `[CLS]` vs mean vs max vs last-token? Re-check the model card.
7. **Reranker** — is your cross-encoder reranker still aligned with the bi-encoder embed? (Common drift when one is upgraded but not the other.)

> 🎓 **The takeaway.** Embeddings are not magic dust — they are **structured high-dimensional geometry** that you can diagnose, compare, align, and steer with the same ML toolkit you'd use anywhere else. Once you internalize cosine + RSA + axis projection + whitening, half the mystery of RAG, multimodal retrieval, and embedding bias evaporates.

## ✅ Recap

- **Token embedding** = static `[V × D]` lookup table; **contextual embedding** = layer-`l` activations; **pooled embedding** = single vector per text.
- **All-to-all cosine matrix** reveals block-diagonal semantic categories; random pairs are at ~0.3 (anisotropy), not 0.
- **Sequential cosine** highlights content vs function words; numbers form a uniform ladder.
- **Cosine graphs** at threshold τ → community detection → bootstrap KGs (M87 GraphRAG callback).
- **RSA** compares two models' geometries via similarity-matrix correlation on shared stimuli.
- **Linear semantic axes** quantify gender / sentiment / size / formality directions; M100 bias measurement.
- **Analogy vectors** sometimes work (king→queen) and sometimes fail; failures reveal anisotropy + polysemy + bias.
- **Anisotropy** is universal; **whitening / all-but-the-top / contrastive fine-tune** are the fixes.
- **Procrustes / CCA / linear** align two embedding spaces given anchors — the embedding-migration toolbox.
- **2026 stack**: OpenAI `text-embed-3-*` · Voyage-3 · BGE-M3 · Nomic v1.5 · CLIP/SigLIP for multimodal · Matryoshka-truncation aware.

Next: **M103 — Output-Layer & Decoding Analysis** (Cohen Ch4).
